# CrocoDashChecks

CrocoDash `main`, checked end to end on NCAR's systems.
[How it works](how_it_works.md) · [Performance](performance.ipynb)

```{toctree}
:hidden:
performance
how_it_works
integration
```

In [ ]:
import checks
from checks import FAIL, NOT_RUN, OK, WARN

wf = checks.workflow()  # weekly: the grids and forcings on every grid in the matrix
runs = checks.load(checks.CESM_RUNS)  # weekly: CESM cases on the grids marked for them
health = checks.load(checks.HEALTH)  # daily: every data access method
grids = checks.matrix()

In [ ]:
def summary_row(name, kind, results, counts):
    """A check: how many of its parts pass, when it ran, and the CrocoDash it tested."""
    if results is None:
        return [name, f"{WARN} No results yet", "", ""]
    ok, total = counts(results)
    result = f"{OK if ok == total else FAIL} {ok}/{total} pass"
    if checks.is_stale(results["date"], kind):
        result += f" ({WARN} {checks.age_days(results['date'])} days old)"
    crocodash = checks.commit(results.get("crocodash"), "CrocoDash")
    return [name, result, checks.when(results["date"]), crocodash]


def workflow_counts(wf):
    marks = [mark for mark, _ in wf["outcomes"].values()]
    return marks.count(OK), len(marks)


checks.show(
    checks.table(
        ["Check", "Result", "Last run", "CrocoDash"],
        [
            summary_row("[Workflow](#matrix)", "workflow", wf, workflow_counts),
            summary_row("[CESM runs](#cesm)", "cesm", runs, checks.cesm_counts),
            summary_row(
                "[Data access](#data-access)", "health", health, checks.health_counts
            ),
        ],
    )
)

(matrix)=
## Test matrix

One row per grid in
[`matrix.yaml`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/integration/matrix.yaml);
blank is off.

In [ ]:
outcomes = wf["outcomes"] if wf else {}


def mark(test_id):
    """A workflow step's mark: passed, failed, skipped, or no result yet."""
    return outcomes.get(test_id, (WARN, ""))[0]


def tally(marks):
    """'✅ 9/9', or ❌ with how many passed."""
    if not marks:
        return WARN
    return f"{OK if marks.count(OK) == len(marks) else FAIL} {marks.count(OK)}/{len(marks)}"


def cesm(name):
    """How many CESM tests on this grid passed."""
    tests = [t for t in (runs or {}).get("tests", []) if t.get("grid") == name]
    return tally([OK if t["passed"] else FAIL for t in tests])


def sweep(test):
    """How many cases of a Performance sweep passed."""
    return tally([m for t, (m, _) in outcomes.items() if t.startswith(test + "[")])


rows = []
for name, grid in grids.items():
    forcings = grid.get("forcings", [])
    rows.append(
        [
            name,
            checks.resolution(grid["grid"]),
            mark(f"test_grid[{name}]"),
            mark(f"test_bc[{name}]") if "bc" in forcings else "",
            mark(f"test_runoff[{name}]") if "runoff" in forcings else "",
            cesm(name) if grid.get("cesm_runs") else "",
        ]
    )
rows.append(
    [
        "[Performance sweeps](performance.ipynb)",
        "",
        "",
        sweep("test_bc_sweep"),
        sweep("test_runoff_sweep"),
        "",
    ]
)
text = checks.table(["Grid", "Res.", "Grid & topo", "OBC", "Runoff", "CESM"], rows)
text += f"\n\n{OK} pass · {FAIL} fail · {NOT_RUN} skipped · {WARN} no result yet"
failed = [
    f"- `{t}`: {checks.plain(why)}" for t, (m, why) in outcomes.items() if m == FAIL
]
if failed:
    text += "\n\n" + "\n".join(failed)
checks.show(text)

(cesm)=
## CESM runs

MOM_interface's regional SMS tests, built with CrocoDash and run on Derecho, weekly.

In [ ]:
def phase_marks(test):
    """One mark per CIME phase, in order: passed, failed or not reached."""
    status = dict(test["phases"])
    marks = {"PASS": OK, "FAIL": FAIL}
    return "".join(marks.get(status.get(phase), NOT_RUN) for phase, _ in checks.PHASES)


def result(test):
    """'Pass', or the first phase that failed."""
    if test["passed"]:
        return f"{OK} Pass"
    status = dict(test["phases"])
    failed = [label for phase, label in checks.PHASES if status.get(phase) == "FAIL"]
    return f"{FAIL} {failed[0]}" if failed else f"{WARN} Incomplete"


if runs is None:
    checks.show("No results yet.")
else:
    # Failures first. Runs before the test matrix didn't record a grid.
    tests = sorted(
        runs["tests"], key=lambda t: (t["passed"], t.get("grid", ""), t["compset"])
    )
    rows = [
        [
            f"`{t['compset']}`",
            t.get("grid", "—"),
            f"`{t['test']}`",
            phase_marks(t),
            result(t),
        ]
        for t in tests
    ]
    order = " → ".join(label for _, label in checks.PHASES)
    checks.show(
        checks.table(["Compset", "Grid", "Test", "Phases", "Result"], rows)
        + f"\n\nPhases: {order}. CESM {checks.commit(runs.get('cesm'), 'CESM')}."
    )

(data-access)=
## Data access

Every way CrocoDash fetches data, called on Casper, daily.

In [ ]:
if health is None:
    checks.show("No results yet.")
else:
    by_name = sorted(
        health["validate_checks"], key=lambda c: (c["product"], c["method"])
    )
    failing = [c for c in by_name if not c["ok"]]
    working = [c for c in by_name if c["ok"]]
    text = ""
    if failing:
        text += f"**{FAIL} Failing ({len(failing)})**\n\n" + checks.table(
            ["Product", "Access method", "Why", "Time"],
            [
                [
                    c["product"],
                    f"`{c['method']}`",
                    checks.plain(c.get("detail", "")),
                    checks.fmt_s(c["elapsed_s"]),
                ]
                for c in failing
            ],
        )
        text += "\n\n"
    text += f"**{OK} Working ({len(working)})**\n\n" + checks.table(
        ["Product", "Access method", "Time"],
        [
            [c["product"], f"`{c['method']}`", checks.fmt_s(c["elapsed_s"])]
            for c in working
        ],
    )
    checks.show(text)

(performance-summary)=
(mom6-scaling)=
## Performance

[Timings and fits](performance.ipynb), including MOM6's speed against its task count.